In [1]:
pip install scikit-optimize

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from collections import Counter

# Modelos, Métricas y Preprocesamiento
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Optimización Bayesiana
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

In [4]:
# --- Carga de Datos ---
try:
    df = pd.read_csv('dataset_limpio.csv')
    print("Datos cargados exitosamente. Forma:", df.shape)
    
    # Limpieza: Eliminar event_id si existe
    if 'event_id' in df.columns:
        df = df.drop('event_id', axis=1)

    target_col = 'particle_type'
    X = df.drop(target_col, axis=1)
    y_str = df[target_col]

    # Codificación de la variable objetivo a números
    classes = sorted(y_str.unique())
    y = pd.Series(y_str.factorize(sort=True)[0], name=target_col)

    print("\nConteo de clases:")
    print(y_str.value_counts())
    print("\nClases mapeadas:", {i: cls for i, cls in enumerate(classes)})

    # División en entrenamiento y prueba (estratificada)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    print(f"\nEntrenamiento: {X_train.shape}, Prueba: {X_test.shape}")

except FileNotFoundError:
    print("ERROR: El archivo 'dataset_limpio.csv' no fue encontrado.")

Datos cargados exitosamente. Forma: (30905, 9)

Conteo de clases:
particle_type
track       30111
muon          397
photon        251
electron      146
Name: count, dtype: int64

Clases mapeadas: {0: 'electron', 1: 'muon', 2: 'photon', 3: 'track'}

Entrenamiento: (23178, 7), Prueba: (7727, 7)


In [5]:
%%time

# 1. Definición del Pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42, probability=True))
])

# 2. Definición del espacio de búsqueda bayesiano (BO-GP)
# Buscamos en rangos continuos para C y Gamma
search_space = {
    'svm__C': Real(0.1, 100, prior='log-uniform'),
    'svm__gamma': Real(0.001, 1, prior='log-uniform'),
    'svm__kernel': Categorical(['rbf', 'poly', 'sigmoid'])
}

# 3. Inicialización de BayesSearchCV
# n_iter=20 define cuántas combinaciones inteligentes probará
opt = BayesSearchCV(
    svm_pipeline,
    search_space,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    random_state=42,
    scoring='accuracy',
    verbose=0
)

print("\nIniciando Optimización Bayesiana (BO-GP) sobre SVM...")
start_time = time.time()
opt.fit(X_train, y_train)
end_time = time.time()

# 4. Resultados del mejor modelo
best_svm_model = opt.best_estimator_
print(f"Tiempo de optimización: {end_time - start_time:.2f} segundos")
print("Mejores parámetros:", opt.best_params_)
print(f"Mejor precisión en validación: {opt.best_score_:.4f}")

# Predicciones finales con el mejor modelo (una sola vez)
y_pred = best_svm_model.predict(X_test)
y_score = best_svm_model.predict_proba(X_test)


Iniciando Optimización Bayesiana (BO-GP) sobre SVM...
CPU times: total: 1min 45s
Wall time: 14h 27min 53s


KeyboardInterrupt: 

In [1]:
# --- Reporte de Clasificación ---
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred, target_names=classes))

# --- Matriz de Confusión ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

plt.figure(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Matriz de Confusión (SVM + BO-GP)')
plt.grid(False)
plt.show()

# --- Precisión por Clase (Recall) ---
print("\n--- Sensibilidad (Recall) por Clase ---")
class_recall = cm.diagonal() / cm.sum(axis=1)
for i, cls in enumerate(classes):
    print(f"Clase {cls}: {class_recall[i]:.4f}")


--- Reporte de Clasificación ---


NameError: name 'classification_report' is not defined

In [ ]:
# Binarizar etiquetas para ROC multiclase
y_test_bin = label_binarize(y_test, classes=range(len(classes)))
n_classes = len(classes)

plt.figure(figsize=(10, 8))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {classes[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.50)')
plt.xlabel('FPR (Falsos Positivos)')
plt.ylabel('TPR (Recall / Verdaderos Positivos)')
plt.title('Curva ROC - Enfoque One-vs-Rest (Mejor SVM Bayesiano)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
print("\nCalculando Curva de Aprendizaje...")

train_sizes, train_scores, test_scores = learning_curve(
    best_svm_model, 
    X_train, 
    y_train, 
    cv=5, 
    n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)

# Medias
train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(10, 8))
plt.plot(train_sizes, train_mean, 'o-', color="darkred", label="Score Entrenamiento")
plt.plot(train_sizes, test_mean, 'o-', color="darkgreen", label="Score Validación")

plt.title('Curva de Aprendizaje (Mejor SVM - BO-GP)')
plt.xlabel('Tamaño de la Muestra de Entrenamiento')
plt.ylabel('Puntuación Accuracy')
plt.legend(loc="best")
plt.grid(True)
plt.show()

print("\nAnálisis completado.")

## SIN TRACKS

In [ ]:
# --- Carga de Datos ---
try:
    df = pd.read_csv("dataset_limpio_notracks.csv")
    print("Datos cargados exitosamente. Forma:", df.shape)
    
    # Limpieza: Eliminar event_id si existe
    if 'event_id' in df.columns:
        df = df.drop('event_id', axis=1)

    target_col = 'particle_type'
    X = df.drop(target_col, axis=1)
    y_str = df[target_col]

    # Codificación de la variable objetivo a números
    classes = sorted(y_str.unique())
    y = pd.Series(y_str.factorize(sort=True)[0], name=target_col)

    print("\nConteo de clases:")
    print(y_str.value_counts())
    print("\nClases mapeadas:", {i: cls for i, cls in enumerate(classes)})

    # División en entrenamiento y prueba (estratificada)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    print(f"\nEntrenamiento: {X_train.shape}, Prueba: {X_test.shape}")

except FileNotFoundError:
    print("ERROR: El archivo 'dataset_limpio.csv' no fue encontrado.")

In [ ]:
%%time

# 1. Definición del Pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42, probability=True))
])

# 2. Definición del espacio de búsqueda bayesiano (BO-GP)
# Buscamos en rangos continuos para C y Gamma
search_space = {
    'svm__C': Real(0.1, 100, prior='log-uniform'),
    'svm__gamma': Real(0.001, 1, prior='log-uniform'),
    'svm__kernel': Categorical(['rbf', 'poly', 'sigmoid'])
}

# 3. Inicialización de BayesSearchCV
# n_iter=20 define cuántas combinaciones inteligentes probará
opt = BayesSearchCV(
    svm_pipeline,
    search_space,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    random_state=42,
    scoring='accuracy',
    verbose=0
)

print("\nIniciando Optimización Bayesiana (BO-GP) sobre SVM...")
start_time = time.time()
opt.fit(X_train, y_train)
end_time = time.time()

# 4. Resultados del mejor modelo
best_svm_model = opt.best_estimator_
print(f"Tiempo de optimización: {end_time - start_time:.2f} segundos")
print("Mejores parámetros:", opt.best_params_)
print(f"Mejor precisión en validación: {opt.best_score_:.4f}")

# Predicciones finales con el mejor modelo (una sola vez)
y_pred = best_svm_model.predict(X_test)
y_score = best_svm_model.predict_proba(X_test)

In [ ]:
# --- Reporte de Clasificación ---
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred, target_names=classes))

# --- Matriz de Confusión ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

plt.figure(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Matriz de Confusión (SVM + BO-GP)')
plt.grid(False)
plt.show()

# --- Precisión por Clase (Recall) ---
print("\n--- Sensibilidad (Recall) por Clase ---")
class_recall = cm.diagonal() / cm.sum(axis=1)
for i, cls in enumerate(classes):
    print(f"Clase {cls}: {class_recall[i]:.4f}")

In [ ]:
# Binarizar etiquetas para ROC multiclase
y_test_bin = label_binarize(y_test, classes=range(len(classes)))
n_classes = len(classes)

plt.figure(figsize=(10, 8))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Clase {classes[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.50)')
plt.xlabel('FPR (Falsos Positivos)')
plt.ylabel('TPR (Recall / Verdaderos Positivos)')
plt.title('Curva ROC - Enfoque One-vs-Rest (Mejor SVM Bayesiano)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
print("\nCalculando Curva de Aprendizaje...")

train_sizes, train_scores, test_scores = learning_curve(
    best_svm_model, 
    X_train, 
    y_train, 
    cv=5, 
    n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)

# Medias
train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(10, 8))
plt.plot(train_sizes, train_mean, 'o-', color="darkred", label="Score Entrenamiento")
plt.plot(train_sizes, test_mean, 'o-', color="darkgreen", label="Score Validación")

plt.title('Curva de Aprendizaje (Mejor SVM - BO-GP)')
plt.xlabel('Tamaño de la Muestra de Entrenamiento')
plt.ylabel('Puntuación Accuracy')
plt.legend(loc="best")
plt.grid(True)
plt.show()

print("\nAnálisis completado.")